In [15]:
%pip install -q langchain-ollama

Note: you may need to restart the kernel to use updated packages.


In [14]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2:1b")

llm.invoke("what is the capital of France?") # prompt: LLM 호출할 때 쓰는 명령어


AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-06-09T05:31:03.792492Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1090020291, 'load_duration': 976101166, 'prompt_eval_count': 32, 'prompt_eval_duration': 43053000, 'eval_count': 8, 'eval_duration': 69562000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--019eaadc-e0ad-7b91-9c9a-d6889549b8ec-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 8, 'total_tokens': 40})

In [15]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt_template = PromptTemplate(
    template="what is the capital of {country}? Return the name of the city only.",
    input_variables=["country"],
)

prompt = prompt_template.invoke({"country": "France"})

print(prompt)

ai_message = llm.invoke(prompt_template.invoke({"country": "France"}))

print(ai_message)

output_parser = StrOutputParser()

answer = output_parser.invoke(llm.invoke(prompt_template.invoke({"country": "France"})))

print(answer)

text='what is the capital of France? Return the name of the city only.'
content='Paris.' additional_kwargs={} response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-06-09T05:31:05.185496Z', 'done': True, 'done_reason': 'stop', 'total_duration': 223823875, 'load_duration': 163125959, 'prompt_eval_count': 40, 'prompt_eval_duration': 38443000, 'eval_count': 3, 'eval_duration': 20593000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'} id='lc_run--019eaadc-e980-7b53-b50d-40e8e9a7b40d-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 40, 'output_tokens': 3, 'total_tokens': 43}
Paris


In [8]:
ai_message

AIMessage(content='Paris', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-06-09T05:21:08.721929Z', 'done': True, 'done_reason': 'stop', 'total_duration': 271615000, 'load_duration': 209438375, 'prompt_eval_count': 40, 'prompt_eval_duration': 50220000, 'eval_count': 2, 'eval_duration': 10452000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--019eaad3-cf61-7ca0-9c8b-0a5eac7988be-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 40, 'output_tokens': 2, 'total_tokens': 42})

In [9]:
answer

'Paris'

In [18]:
from pydantic import BaseModel, Field

class CountryDetail(BaseModel):
    capital: str = Field(description="The capital of the country")
    population: int = Field(description="The population of the country")
    area: int = Field(description="The area of the country")
    language: str = Field(description="The language of the country")
    currency: str = Field(description="The currency of the country")

structured_llm = llm.with_structured_output(CountryDetail)

structured_llm.invoke(prompt_template.invoke({"country": "France"}))


CountryDetail(capital='Paris', population=13000, area=1018, language=':', currency='EUR')

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

country_detail_prompt = PromptTemplate(
    template="""Give following information about {country}:
    - capital
    - population
    - area
    - language
    - currency
    return in json format. and return the json dictionary only.
    """,
    input_variables=["country"],
)

country_detail_prompt.invoke({"country": "France"})

json_ai_message = structured_llm.invoke(country_detail_prompt.invoke({"country": "France"}))

# JsonOutputParser는 Str 을 return 함
# output_parser = JsonOutputParser()
# output_parser.invoke(llm.invoke(country_detail_prompt.invoke({"country": "France"})))


In [20]:
json_ai_message

CountryDetail(capital='Paris', population=67, area=643, language='French', currency='Euros')

In [24]:
json_ai_message.model_dump()

{'capital': 'Paris',
 'population': 67,
 'area': 643,
 'language': 'French',
 'currency': 'Euros'}

In [25]:
json_ai_message.model_dump()['capital']

'Paris'